# Metadata Automation for Repositories (DSpace / OpenAIRE)

**Objective:** Transform raw metadata (CSV) into an ingestion-ready format for DSpace and ensure compatibility with OpenAIRE 4.0 and FAIR principles.

This notebook demonstrates:
- Data cleaning and normalization
- Mapping to Dublin Core and DSpace schemas
- FAIR principle validation
- Export of cleaned data

Author: Luis Enrique Lescano Borrego  
GitHub: [luislescano](https://github.com/luislescano)


## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

# For FAIR validation we'll check for required fields
REQUIRED_FIELDS = [
    'title',
    'author',
    'date',
    'type',
    'identifier',   # DOI or handle
    'rights'        # license
]

## 2. Load Sample Metadata

We'll create a small sample dataset representing theses and articles from a university. In practice, you would load a CSV file using `pd.read_csv('raw_metadata.csv')`.

In [ ]:
# Create sample data
data = {
    'title': [
        'Impact of AI on research data management',
        'Open science practices in Latin America',
        'FAIR data in biodiversity research',
        'Digital libraries: a case study'
    ],
    'author': [
        'Luis Lescano; Maria Gomez',
        'Ana Rodriguez',
        'Carlos Perez; Sofia Mendez',
        'Juan Carlos'  # incomplete
    ],
    'date': [
        '2025-03-01',
        '2024',
        '2023-12-15',
        '2023'
    ],
    'type': [
        'article',
        'thesis',
        'dataset',
        'book'
    ],
    'identifier': [
        '10.1234/ai.rdm.2025',
        'handle:12345',
        '',
        'https://doi.org/10.5678/dlib.2023'
    ],
    'rights': [
        'CC BY 4.0',
        'All rights reserved',
        'CC0',
        ''
    ],
    'language': [
        'en',
        'es',
        'en',
        'es'
    ]
}

df_raw = pd.DataFrame(data)
df_raw.head()

## 3. Data Cleaning and Normalization

### 3.1 Author names: split and standardize
We'll convert semicolon-separated authors into a list format that DSpace can ingest.

In [ ]:
def split_authors(author_str):
    if pd.isna(author_str):
        return []
    # Split by semicolon and strip whitespace
    authors = [a.strip() for a in author_str.split(';')]
    # Capitalize each name part? Not necessary, but we can normalize
    return authors

df_raw['authors_list'] = df_raw['author'].apply(split_authors)
df_raw[['author', 'authors_list']].head()

### 3.2 Date normalization: ensure YYYY-MM-DD or just year

In [ ]:
def normalize_date(date_str):
    if pd.isna(date_str):
        return ''
    # Try to parse as full date
    try:
        d = pd.to_datetime(date_str)
        return d.strftime('%Y-%m-%d')
    except:
        # If only year, keep as is
        if re.match(r'^\d{4}$', str(date_str)):
            return str(date_str)
        else:
            return ''

df_raw['date_norm'] = df_raw['date'].apply(normalize_date)
df_raw[['date', 'date_norm']].head()

### 3.3 Identifier normalization: ensure DOI is extracted

In [ ]:
def extract_doi(identifier):
    if pd.isna(identifier):
        return ''
    # Look for DOI pattern
    match = re.search(r'10\.\d{4,9}/[-._;()/:A-Z0-9]+', str(identifier), re.I)
    if match:
        return match.group()
    # If not found, return original (maybe handle)
    return identifier

df_raw['doi'] = df_raw['identifier'].apply(extract_doi)
df_raw[['identifier', 'doi']].head()

## 4. Map to DSpace Ingestion Schema

DSpace expects a specific CSV format for bulk import (e.g., `dc.title`, `dc.contributor.author`, etc.). We'll create a new DataFrame with the required fields.

We'll map:
- `dc.title` → title
- `dc.contributor.author` → authors_list (joined by `||`)
- `dc.date.issued` → date_norm
- `dc.type` → type
- `dc.identifier` → identifier
- `dc.rights` → rights
- `dc.language.iso` → language

Additionally, we'll add `dc.identifier.uri` if DOI is present (we could also add handle).

In [ ]:
dspace_mapping = {
    'dc.title': df_raw['title'],
    'dc.contributor.author': df_raw['authors_list'].apply(lambda x: '||'.join(x)),
    'dc.date.issued': df_raw['date_norm'],
    'dc.type': df_raw['type'],
    'dc.identifier': df_raw['identifier'],
    'dc.rights': df_raw['rights'],
    'dc.language.iso': df_raw['language'],
    'dc.identifier.doi': df_raw['doi']
}

df_dspace = pd.DataFrame(dspace_mapping)
df_dspace.head()

## 5. FAIR Principles Validation

We'll check each record for the presence of essential elements for FAIRness.

- **Findable**: have a persistent identifier (DOI or handle)
- **Accessible**: metadata is open and at least one access link (we'll consider rights statement)
- **Interoperable**: uses standard metadata (we already have Dublin Core mapping)
- **Reusable**: clear license (rights)

In [ ]:
def check_fair(row):
    issues = []
    if not row['doi'] and not ('handle' in str(row['dc.identifier']).lower()):
        issues.append('Missing persistent identifier (DOI/handle)')
    if not row['dc.rights'] or row['dc.rights'] == 'All rights reserved':
        issues.append('Missing or restrictive license')
    if not row['dc.date.issued']:
        issues.append('Missing publication date')
    if not row['dc.title']:
        issues.append('Missing title')
    return ', '.join(issues) if issues else 'OK'

df_dspace['FAIR_check'] = df_dspace.apply(check_fair, axis=1)
df_dspace[['dc.title', 'dc.identifier.doi', 'dc.rights', 'FAIR_check']]

## 6. Export Cleaned Data

Finally, we save the transformed data as a CSV file ready for DSpace bulk import.

In [ ]:
output_file = 'dspace_ingest_ready.csv'
df_dspace.to_csv(output_file, index=False)
print(f"Data saved to {output_file}")

# Also print a summary
print("\nFAIR compliance summary:")
print(df_dspace['FAIR_check'].value_counts())

## 7. Conclusion

This notebook demonstrates a reusable pipeline for:
- Cleaning raw metadata
- Mapping to DSpace and OpenAIRE schemas
- Validating FAIR compliance
- Producing an ingest-ready CSV

The same approach can be extended to other repositories (Koha, OJS) by adjusting the target schema.

---

*This work is part of a portfolio showcasing expertise in research data management and metadata automation.*